### 환경설정
- chroma(벡터Db) : 로컬 인메모리 모드지원, 서버불필요
- Neo4j python driver : Neo4j 서버통신
- sentence-transformers : 텍스트 임베딩
- sckit-learn : PCA 데이터 분석 

In [ ]:
!pip install chromadb sentence-transformers neo4j matplotlib networkx scikit-learn -q

In [2]:
# 설치확인
import chromadb
import neo4j
import sentence_transformers
print(chromadb.__version__, neo4j.__version__, sentence_transformers.__version__)


1.5.9 6.2.0 5.5.1


### 벡터DB
- 고차원벡터(Embedding)를 저장, 유사도검색을 수행하는데 특화
- RAG 핵심 인프라 : LLM의 외부 지식검색에 필수적
- 메타데이터 필터링 : 벡터검색 + 조건 필터링 동시 지원

### 벡터DB 종류
- Chroma : 경량                : 학습(프로토타입)
- Pinecone : 관리형 ,고 가용성   : 대용량 프로젝트
- Weaviate : 멀티모달           : 복합검색
- Milvus : 분선처리 GPU         : 대용량 프로젝트
- FAISS : Meta 오픈소스, 초고속  : 연구용(벤치마크)

In [3]:
import chromadb
client = chromadb.Client()  # in-memory
# client = chromadb.PersistentClient(path='./outputs')
client.heartbeat()  # 

1779759801272309400

### collection  생성
- 벡터들의 논리적 그룹(RDBMS 테이블 개념)
- 거리함수 : cosine(기본)
client

In [4]:
collection = client.get_or_create_collection(
    name = 'forean_foods',
    metadata={'hnsw:space':'cosine'}
)
print(f'컬렉션 이름 : {collection.name}') 
print(f'현재 문서 수 : {collection.count()}')

컬렉션 이름 : forean_foods
현재 문서 수 : 0


In [5]:
documents = [
    "김치찌개는 돼지고기와 김치를 넣고 끓인 한국의 대표적인 찌개 요리입니다.",
    "불고기는 간장 양념에 재운 소고기를 구워 먹는 한국 전통 요리입니다.",
    "비빔밥은 밥 위에 다양한 나물과 고추장을 넣고 비벼 먹는 음식입니다.",
    "된장찌개는 된장을 풀어 두부, 감자, 호박 등을 넣고 끓인 찌개입니다.",
    "삼겹살은 돼지 뱃살을 구워 쌈 채소에 싸서 먹는 인기 있는 요리입니다.",
    "떡볶이는 떡과 어묵을 고추장 양념에 볶아 만든 한국의 길거리 음식입니다.",
    "냉면은 메밀 면을 차가운 육수에 말아 먹는 여름철 별미입니다.",
    "잡채는 당면에 다양한 채소와 고기를 볶아 만든 명절 음식입니다.",
    "갈비탕은 소갈비를 오랫동안 끓여 만든 깊은 맛의 탕 요리입니다.",
    "순두부찌개는 부드러운 순두부에 해물이나 고기를 넣어 끓인 매운 찌개입니다.",
]

metadatas = [
    {"category": "찌개", "main_ingredient": "돼지고기", "spicy": True},
    {"category": "구이", "main_ingredient": "소고기", "spicy": False},
    {"category": "밥",  "main_ingredient": "채소",   "spicy": True},
    {"category": "찌개", "main_ingredient": "된장",   "spicy": False},
    {"category": "구이", "main_ingredient": "돼지고기", "spicy": False},
    {"category": "분식", "main_ingredient": "떡",     "spicy": True},
    {"category": "면",  "main_ingredient": "메밀",   "spicy": False},
    {"category": "볶음", "main_ingredient": "당면",   "spicy": False},
    {"category": "탕",  "main_ingredient": "소고기", "spicy": False},
    {"category": "찌개", "main_ingredient": "순두부", "spicy": True},
]

ids = [f'food_{i:03d}' for i in range(len(documents))]
print(f'ids = {ids}')

collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)
print(f'{collection.count()}개 문서 추가 완료')
for doc_id, doc in zip(ids,documents):
    print(f'    {doc_id} : {doc[:30]}...')

ids = ['food_000', 'food_001', 'food_002', 'food_003', 'food_004', 'food_005', 'food_006', 'food_007', 'food_008', 'food_009']


C:\Users\Playdata\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:09<00:00, 8.63MiB/s]


10개 문서 추가 완료
    food_000 : 김치찌개는 돼지고기와 김치를 넣고 끓인 한국의 대표적인 찌개 요리입니다.
    food_001 : 불고기는 간장 양념에 재운 소고기를 구워 먹는 한국 전통 요리입니다.
    food_002 : 비빔밥은 밥 위에 다양한 나물과 고추장을 넣고 비벼 먹는 음식입니다.
    food_003 : 된장찌개는 된장을 풀어 두부, 감자, 호박 등을 넣고 끓인 찌개입니다.
    food_004 : 삼겹살은 돼지 뱃살을 구워 쌈 채소에 싸서 먹는 인기 있는 요리입니다.
    food_005 : 떡볶이는 떡과 어묵을 고추장 양념에 볶아 만든 한국의 길거리 음식입니다.
    food_006 : 냉면은 메밀 면을 차가운 육수에 말아 먹는 여름철 별미입니다.
    food_007 : 잡채는 당면에 다양한 채소와 고기를 볶아 만든 명절 음식입니다.
    food_008 : 갈비탕은 소갈비를 오랫동안 끓여 만든 깊은 맛의 탕 요리입니다.
    food_009 : 순두부찌개는 부드러운 순두부에 해물이나 고기를 넣어 끓인 매운 찌개입니다.


### 유사도 검색

In [7]:
query  = '매운 국물 요리가 먹고 싶어요'
results = collection.query(
    query_texts=[query],
    n_results=5  # top_5
)
for doc, meta,dist in zip(results['documents'][0],results['metadatas'][0],results['distances'][0]):
    similarity = 1- dist # 코사인거리->유사도 변환
    print(doc,meta,dist)

떡볶이는 떡과 어묵을 고추장 양념에 볶아 만든 한국의 길거리 음식입니다. {'category': '분식', 'spicy': True, 'main_ingredient': '떡'} 0.15752524137496948
순두부찌개는 부드러운 순두부에 해물이나 고기를 넣어 끓인 매운 찌개입니다. {'main_ingredient': '순두부', 'spicy': True, 'category': '찌개'} 0.19012266397476196
김치찌개는 돼지고기와 김치를 넣고 끓인 한국의 대표적인 찌개 요리입니다. {'spicy': True, 'main_ingredient': '돼지고기', 'category': '찌개'} 0.21276319026947021
냉면은 메밀 면을 차가운 육수에 말아 먹는 여름철 별미입니다. {'spicy': False, 'category': '면', 'main_ingredient': '메밀'} 0.2229100465774536
잡채는 당면에 다양한 채소와 고기를 볶아 만든 명절 음식입니다. {'spicy': False, 'main_ingredient': '당면', 'category': '볶음'} 0.25143176317214966
